# Imports

In [ ]:
# Data Management
import pandas as pd
import numpy as np
import re
from pandas_datareader.data import DataReader
from ta import add_all_ta_features

# Statistics
from statsmodels.tsa.stattools import adfuller

# Data Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

# Supervised Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RepeatedKFold

# Reporting
import matplotlib.pyplot as plt

# Data Ingestion

In [ ]:
df = pd.read_csv("sydney-house-prices.csv")

In [ ]:
print(f"Amount of rows: {df.shape[0]}")
print(f"Amount of columns: {df.shape[1]}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Function to unify column names
def unify_column_names(name: str) -> str:
    return re.sub(r'([a-z])([A-Z])', r'\1_\2', name).lower()

In [ ]:
df.columns = df.columns.map(lambda x: unify_column_names(x))

In [ ]:
df.info()

# Feature Engineering

### Handle Non-numerical Data

In [ ]:
# Count unique items for 'suburb'
suburb_unique_text = df['suburb'].nunique()
print(f'Number of unique suburbs: {suburb_unique_text}')
print("Will require label encoding.")

In [ ]:
# Perform label encoding and assign a number to each 'suburb'
label_encoder = LabelEncoder()
encoded_suburbs = label_encoder.fit_transform(df['suburb'])
df['suburbs_encoded'] = encoded_suburbs
df.head(40)

In [ ]:
# Count unique items for 'prop_type'
prop_type_unique_text = df['prop_type'].nunique()
print(f'Number of unique property types: {prop_type_unique_text}')
print("Will require one hot encoding.")

In [ ]:
# Perform one hot encoding and assign a number to each 'prop_type'
one_hot_encoder = OneHotEncoder(sparse_output=False) # Use sparse=False to get a dense array (not sparse matrix)

# Reshape the 'prop_type' column to be a 2D array
encoded_property_types = one_hot_encoder.fit_transform(df['prop_type'].values.reshape(-1, 1))

# Convert the result to a DataFrame from a Series
encoded_property_types_df = pd.DataFrame(encoded_property_types, columns=one_hot_encoder.categories_[0])

# Append the encoded columns to the original dataframe
df = pd.concat([df, encoded_property_types_df], axis=1)
df.head(40)

### Set Target

In [ ]:
# Set target
df['target'] = df['sell_price']
df.head(20)

### Remove Redundant Columns / Features

In [ ]:
# Remove features
df_drop = df.copy()
df_drop.drop(columns=['date', 'id', 'suburb', 'prop_type', 'sell_price'], inplace=True)
df_drop.head(20)

### Check for Missing & Outlier Values

In [ ]:
# Check for NaN, None, or NA values
print(f'Total number of missing values: {df_drop.isna().sum().sum()}')

In [ ]:
# Check for Infinite Values
print(f'Total number of infinite values: {df_drop.isin([np.inf, -np.inf]).sum().sum()}')

In [ ]:
# Handle missing values by filling them in with the column mean
df_drop = df_drop.fillna(df_drop.mean())

In [ ]:
# Check again if there are missing values to make sure they were filled with the column mean
print(f'Total number of missing values: {df_drop.isna().sum().sum()}')